<a href="https://colab.research.google.com/github/lbenit/Floristic_map_africa/blob/main/03_Determine_indicators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Read me
This script runs in python, but utlises R to do the indicator analysis. This approach allows for easily integrating this script with the rest of the workflow while using R packags for the analysis. Indicators are identified separately for clsuter k=3 to k=7.

# Set up github connection

In [1]:
!git clone https://github.com/lbenit/Floristic_map_africa.git


Cloning into 'Floristic_map_africa'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 45 (delta 14), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 1.83 MiB | 1.92 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [2]:
%cd Floristic_map_africa


/content/Floristic_map_africa


In [3]:
# Print working directory
import os
print(os.getcwd())

/content/Floristic_map_africa


# Set up R for python

In [4]:

%load_ext rpy2.ipython


In [5]:
%%R
print("R is working")


[1] "R is working"


# Install R packages

In [9]:
%%R
install.packages('indicspecies')
install.packages('vegan')
install.packages('dplyr')
install.packages('reticulate')

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependency ‘permute’

trying URL 'https://cran.rstudio.com/src/contrib/permute_0.9-10.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/indicspecies_1.8.0.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmp46cTQZ/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/vegan_2.7-5.tar.gz'
Content type 'application/x-gzip' length 1451867 bytes (1.4 MB)
downloaded 1.4 MB


The downloaded source packages are in
	‘/tmp/Rtmp46cTQZ/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.1.tar.gz'
Content type 'application/x-gzip' length 923509 bytes (901 KB)
downloaded 901 KB


The downloaded source packages are in
	‘/tmp/Rtmp46cTQZ/downloaded_packages’
Installing package into ‘/u

In [10]:
%%R
# Load packages
library(dplyr)
library(vegan)
library(indicspecies)
library(reticulate)


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Loading required package: permute


# Prepare data

In [17]:
# Load data with clusters for training
%%R
train_cluster<-read.csv("Data/all_data_umap_clusters_parametric.csv")
dim(train_cluster)
#length(unique(train_cluster$plot_id))

[1] 366  21


In [ ]:
%%R
head(train_cluster)

  plot_id      UMAP1     UMAP2      UMAP3 perturbation singularity cluster2
1   ABG_1 -3.1191850 -5.758523 -2.8711393     0.800000   0.7333333        0
2  ABG_10 -3.2932632 -6.026543 -3.2312390     0.800000   1.8000000        0
3  ABG_11 -4.5356507 -6.299250  0.6107124     1.133333   2.0666667        0
4  ABG_12  0.1038292 -4.981802 -2.6310942     1.133333   0.3333333        0
5  ABG_13  0.6025185 -4.934231 -2.8179839     0.600000   0.7333333        0
6  ABG_14 -3.0438986 -5.775639 -3.0868210     1.133333   0.7333333        0
  cluster3 cluster4 cluster5 cluster6 cluster7 cluster8 cluster9 lineage2
1        0        0        1        0        6        6        6        0
2        0        0        1        0        6        6        6        0
3        0        0        4        4        1        0        4        0
4        0        1        0        2        2        2        2        0
5        0        1        0        2        2        2        2        0
6        0        0     

In [7]:
#Load in genus abundance matrix
%%R
df<-read.csv('Data/relative_abundance_data_no_singelton.csv')

In [8]:
%%R
dim(df)

[1] 366  89


In [11]:
%%R
df_train<-df%>%filter(plot_id %in% train_cluster$plot_id)

In [ ]:
%%R
dim(df_train)

[1] 7769  267


In [12]:
%%R
# Match plot names
df_train<-df_train%>%arrange(match(plot_id,train_cluster$plot_id))
rownames(df_train)<-df_train$plot_id

In [13]:
%%R
train<-df_train%>%select(-plot_id)
bi_hell<-decostand(train, method='hellinger')



In [14]:
%%R
dim(train)


[1] 366  88


In [15]:
%%R
dim(bi_hell)

[1] 366  88


In [16]:
%%R
dim(train_cluster)

[1] 366  21


# Indicators for 3 clusters

In [18]:
# Indicator species analysis
%%R
ind <- multipatt(
  x = bi_hell,
  cluster = train_cluster$cluster3,
  func = "r.g"
)


In [19]:
# Look at results
%%R
summary(ind)


 Multilevel pattern analysis
 ---------------------------

 Association function: r.g
 Significance level (alpha): 0.05

 Total number of species: 88
 Selected number of species: 54 
 Number of species associated to 1 group: 47 
 Number of species associated to 2 groups: 7 

 List of species associated to each combination: 

 Group 0  #sps.  9 
               stat p.value   
Combretum     0.536   0.005 **
Guibourtia    0.402   0.005 **
Indet         0.397   0.005 **
Androstachys  0.339   0.005 **
Albizia       0.308   0.005 **
Sclerocarya   0.283   0.005 **
Aganope       0.269   0.005 **
Thespesia     0.154   0.010 **
Dichrostachys 0.153   0.020 * 

 Group 1  #sps.  32 
                    stat p.value   
Brachystegia       0.707   0.005 **
Julbernardia       0.558   0.005 **
Burkea             0.499   0.005 **
Pseudolachnostylis 0.495   0.005 **
Diplorhynchus      0.432   0.005 **
Uapaca             0.416   0.005 **
Parinari           0.354   0.005 **
Bobgunnia          0.340   0.005

In [20]:
# Save the species statss
%%R
ind_table <- ind$sign

In [21]:
%%R
head(ind_table)

             s.0 s.1 s.2 index       stat p.value
Adansonia      1   1   0     4 0.09642396   0.200
Afzelia        1   0   0     1 0.09904926   0.150
Aganope        1   0   0     1 0.26945504   0.005
Albizia        1   0   0     1 0.30801012   0.005
Aloe           0   1   0     2 0.10513276   0.180
Androstachys   1   0   0     1 0.33882368   0.005


In [22]:
# Save data in drive
%%R
write.csv(ind_table, "Data/indicators_3_cluster_parametric.csv")


# Indicators for 4 clusters

In [ ]:
# Indicator species analysis
%%R
ind <- multipatt(
  x = bi_hell,
  cluster = train_cluster$cluster4,
  func = "r.g"
)


In [ ]:
# Look at results
%%R
summary(ind)


 Multilevel pattern analysis
 ---------------------------

 Association function: r.g
 Significance level (alpha): 0.05

 Total number of species: 266
 Selected number of species: 114 
 Number of species associated to 1 group: 74 
 Number of species associated to 2 groups: 39 
 Number of species associated to 3 groups: 1 

 List of species associated to each combination: 

 Group 0  #sps.  19 
                    stat p.value   
Brachystegia       0.769   0.005 **
Julbernardia       0.581   0.005 **
Uapaca             0.392   0.005 **
Parinari           0.343   0.005 **
Pseudolachnostylis 0.311   0.005 **
Pericopsis         0.306   0.005 **
Monotes            0.305   0.005 **
Isoberlinia        0.265   0.005 **
Anisophyllea       0.200   0.005 **
Syzygium           0.189   0.005 **
Faurea             0.188   0.005 **
Bobgunnia          0.185   0.005 **
Cryptosepalum      0.167   0.005 **
Marquesia          0.147   0.005 **
Phyllocosmus       0.104   0.005 **
Erythrina          0.070  

In [ ]:
# Save the species statss
%%R
ind_table <- ind$sign

In [ ]:
%%R
head(ind_table)

             s.0 s.1 s.2 s.3 index       stat p.value
Adansonia      0   1   0   1     9 0.03292568   0.160
Afrocanthium   0   1   0   0     2 0.01985781   0.630
Afzelia        0   1   1   0     8 0.08584007   0.005
Aganope        0   1   0   0     2 0.14031022   0.005
Albizia        1   1   0   0     5 0.17549025   0.005
Alchornea      0   1   0   0     2 0.03971536   0.080


In [ ]:
# Save data in drive
%%R
write.csv(ind_table, "Data/indicators_4_cluster_parametric.csv")


# Get abundance of indicators per plot

In [23]:
%%R
# Look at genera abundances per class for important classes
best_ind<-ind_table%>%filter(p.value<=0.05)
colnames(best_ind)

[1] "s.0"     "s.1"     "s.2"     "index"   "stat"    "p.value"


In [24]:
%%R
cluster<-train_cluster%>%select(plot_id,cluster4,cluster7)
genera<-df_train%>%select(plot_id,any_of(row.names(best_ind)))
together<-left_join(genera,cluster,by="plot_id")
head(together)

  plot_id Aganope   Albizia Androstachys    Annona Antidesma Berchemia
1   MAR_1       0 0.1520658            0 0.0000000         0         0
2  MAR_10       0 0.0000000            0 0.0000000         0         0
3 MAR_100       0 0.0000000            0 0.0000000         0         0
4 MAR_102       0 0.0000000            0 0.0000000         0         0
5 MAR_105       0 0.0000000            0 0.4207271         0         0
6 MAR_106       0 0.0000000            0 0.0000000         0         0
  Bobgunnia Boscia Brachystegia     Burkea Catunaregam Colophospermum
1         0      0    0.0000000 0.00000000           0              0
2         0      0    0.6153157 0.00000000           0              0
3         0      0    0.0000000 0.00000000           0              0
4         0      0    0.7286657 0.02441124           0              0
5         0      0    0.0000000 0.00000000           0              0
6         0      0    0.0000000 0.00000000           0              0
   Combretum 

In [25]:
%%R
cluster4_mean<-together%>%group_by(cluster4)%>%summarise(across(where(is.numeric), mean))
cluster4_sd<-together%>%group_by(cluster4)%>%summarise(across(where(is.numeric), sd))
cluster4_median<-together%>%group_by(cluster4)%>%summarise(across(where(is.numeric), median))
cluster4_sd

# A tibble: 4 × 56
  cluster4 Aganope Albizia Androstachys Annona Antidesma Berchemia Bobgunnia
     <int>   <dbl>   <dbl>        <dbl>  <dbl>     <dbl>     <dbl>     <dbl>
1        0 0.137   0.149         0.0424 0.0900   0          0.0298    0     
2        1 0.0125  0.00365       0      0.0245   0.00274    0.0126    0.0633
3        2 0.00168 0.0127        0.0138 0        0          0         0     
4        3 0.0670  0.120         0.165  0        0          0.0200    0     
# ℹ 48 more variables: Boscia <dbl>, Brachystegia <dbl>, Burkea <dbl>,
#   Catunaregam <dbl>, Colophospermum <dbl>, Combretum <dbl>,
#   Crossopteryx <dbl>, Cussonia <dbl>, Dalbergiella <dbl>,
#   Dichrostachys <dbl>, Diospyros <dbl>, Diplorhynchus <dbl>,
#   Erythrophleum <dbl>, Faurea <dbl>, Ficus <dbl>, Flacourtia <dbl>,
#   Glenniea <dbl>, Guibourtia <dbl>, Hymenocardia <dbl>, Indet <dbl>,
#   Julbernardia <dbl>, Lannea <dbl>, Lecaniodiscus <dbl>, Monotes <dbl>, …


In [ ]:
# Save data in drive
%%R
write.csv(cluster4_mean, "Data/cluster4_abundance_mean.csv")
write.csv(cluster4_median, "Data/cluster4_abundance_median.csv")
write.csv(cluster4_sd, "Data/cluster4_abundance_sd.csv")

# 5 cluster

In [26]:
# Indicator species analysis
%%R
ind <- multipatt(
  x = bi_hell,
  cluster = train_cluster$cluster5,
  func = "r.g"
)

In [27]:
# Look at results
%%R
summary(ind)


 Multilevel pattern analysis
 ---------------------------

 Association function: r.g
 Significance level (alpha): 0.05

 Total number of species: 88
 Selected number of species: 50 
 Number of species associated to 1 group: 41 
 Number of species associated to 2 groups: 8 
 Number of species associated to 3 groups: 1 
 Number of species associated to 4 groups: 0 

 List of species associated to each combination: 

 Group 0  #sps.  20 
                    stat p.value   
Brachystegia       0.728   0.005 **
Julbernardia       0.588   0.005 **
Burkea             0.533   0.005 **
Pseudolachnostylis 0.506   0.005 **
Uapaca             0.448   0.005 **
Parinari           0.383   0.005 **
Bobgunnia          0.368   0.005 **
Pterocarpus        0.300   0.005 **
Monotes            0.297   0.010 **
Faurea             0.246   0.030 * 
Vitex              0.232   0.015 * 
Phyllocosmus       0.227   0.015 * 
Syzygium           0.219   0.005 **
Ochna              0.212   0.020 * 
Zanha              

In [28]:
# Save the species statss
%%R
ind_table <- ind$sign

In [ ]:
# Save data in drive
%%R
write.csv(ind_table, "Data/indicators_5_cluster_parametric.csv")

# 6 cluster

In [29]:
# Indicator species analysis
%%R
ind <- multipatt(
  x = bi_hell,
  cluster = train_cluster$cluster6,
  func = "r.g"
)

In [30]:
# Look at results
%%R
summary(ind)


 Multilevel pattern analysis
 ---------------------------

 Association function: r.g
 Significance level (alpha): 0.05

 Total number of species: 88
 Selected number of species: 52 
 Number of species associated to 1 group: 35 
 Number of species associated to 2 groups: 15 
 Number of species associated to 3 groups: 2 
 Number of species associated to 4 groups: 0 
 Number of species associated to 5 groups: 0 

 List of species associated to each combination: 

 Group 0  #sps.  5 
                stat p.value   
Colophospermum 0.836   0.005 **
Spirostachys   0.441   0.005 **
Glenniea       0.436   0.005 **
Boscia         0.414   0.005 **
Pappea         0.192   0.025 * 

 Group 1  #sps.  4 
               stat p.value   
Guibourtia    0.574   0.005 **
Aganope       0.346   0.005 **
Strychnos     0.249   0.010 **
Dichrostachys 0.218   0.010 **

 Group 2  #sps.  13 
              stat p.value   
Brachystegia 0.839   0.005 **
Julbernardia 0.657   0.005 **
Uapaca       0.524   0.005 **
Par

In [31]:
# Save the species statss
%%R
ind_table <- ind$sign

In [ ]:
# Save data in drive
%%R
write.csv(ind_table, "Data/indicators_6_cluster_parametric.csv")

# 7 Cluster

In [32]:
# Indicator species analysis
%%R
ind <- multipatt(
  x = bi_hell,
  cluster = train_cluster$cluster7,
  func = "r.g"
)

In [33]:
# Look at results
%%R
summary(ind)


 Multilevel pattern analysis
 ---------------------------

 Association function: r.g
 Significance level (alpha): 0.05

 Total number of species: 88
 Selected number of species: 36 
 Number of species associated to 1 group: 25 
 Number of species associated to 2 groups: 9 
 Number of species associated to 3 groups: 2 
 Number of species associated to 4 groups: 0 
 Number of species associated to 5 groups: 0 
 Number of species associated to 6 groups: 0 

 List of species associated to each combination: 

 Group 0  #sps.  8 
              stat p.value   
Brachystegia 0.844   0.005 **
Julbernardia 0.664   0.005 **
Uapaca       0.530   0.005 **
Parinari     0.477   0.005 **
Faurea       0.307   0.020 * 
Monotes      0.306   0.020 * 
Syzygium     0.272   0.015 * 
Ochna        0.264   0.025 * 

 Group 1  #sps.  4 
               stat p.value   
Guibourtia    0.581   0.005 **
Aganope       0.355   0.010 **
Strychnos     0.261   0.035 * 
Dichrostachys 0.229   0.040 * 

 Group 2  #sps.  3 
 

In [34]:
# Save the species statss
%%R
ind_table <- ind$sign

In [ ]:
# Save data in drive
%%R
write.csv(ind_table, "/content/drive/MyDrive/Python_exports/indicators_7_cluster_parametric.csv")

In [ ]:
%%R
# Look at genera abundances per class for important classes
best_ind<-ind_table%>%filter(p.value<=0.05)
head(best_ind)

                s.0 s.1 s.2 s.3 s.4 s.5 s.6 index       stat p.value
Afzelia           0   0   0   1   0   0   0     4 0.09306619   0.005
Aganope           0   0   0   1   0   0   0     4 0.15184417   0.005
Albizia           0   0   0   1   1   0   0    23 0.14874467   0.005
Alchornea         0   0   0   1   0   0   0     4 0.05042791   0.025
Aloe              0   0   0   1   0   0   0     4 0.04419499   0.035
Amblygonocarpus   0   1   1   0   0   0   1    47 0.08603366   0.005


# Get abundances of idnicators for 7 clusters

In [ ]:
%%R
cluster<-train_cluster%>%select(plot_id,cluster4,cluster7)
genera<-df_train%>%select(plot_id,any_of(row.names(best_ind)))
together<-left_join(genera,cluster,by="plot_id")
head(together)

  plot_id Afzelia Aganope     Albizia Alchornea Aloe Amblygonocarpus
1   ABG_1       0       0 0.009626829         0    0               0
2  ABG_10       0       0 0.017013105         0    0               0
3  ABG_11       0       0 0.000000000         0    0               0
4  ABG_12       0       0 0.000000000         0    0               0
5  ABG_13       0       0 0.000000000         0    0               0
6  ABG_14       0       0 0.000000000         0    0               0
  Androstachys Anisophyllea Annona    Baikiaea Balanites      Baphia Bauhinia
1            0            0      0 0.000000000         0 0.000000000        0
2            0            0      0 0.000000000         0 0.000000000        0
3            0            0      0 0.001972722         0 0.002243789        0
4            0            0      0 0.460783009         0 0.025670902        0
5            0            0      0 0.833849429         0 0.010180317        0
6            0            0      0 0.000000000   

In [ ]:
%%R
cluster7_mean<-together%>%group_by(cluster7)%>%summarise(across(where(is.numeric), mean))
cluster7_sd<-together%>%group_by(cluster7)%>%summarise(across(where(is.numeric), sd))
cluster7_median<-together%>%group_by(cluster7)%>%summarise(across(where(is.numeric), median))
cluster7_mean

# A tibble: 7 × 139
  cluster7 Afzelia  Aganope Albizia Alchornea       Aloe Amblygonocarpus
     <int>   <dbl>    <dbl>   <dbl>     <dbl>      <dbl>           <dbl>
1        0 0.00185 0.000780 0.00260 0         0                0.0000392
2        1 0.00243 0.00472  0.0169  0.0000285 0.00000832       0.00197  
3        2 0.00460 0.00511  0.0129  0         0                0.00430  
4        3 0.0122  0.0196   0.0505  0.000409  0.00181          0.000351 
5        4 0.00149 0.000182 0.0176  0         0                0.000109 
6        5 0.00649 0.00518  0.00753 0         0.0000152        0        
7        6 0.00348 0.00278  0.0118  0         0                0.00213  
# ℹ 132 more variables: Androstachys <dbl>, Anisophyllea <dbl>, Annona <dbl>,
#   Baikiaea <dbl>, Balanites <dbl>, Baphia <dbl>, Bauhinia <dbl>,
#   Berchemia <dbl>, Berrya <dbl>, Blighia <dbl>, Bobgunnia <dbl>,
#   Boscia <dbl>, Brachystegia <dbl>, Brackenridgea <dbl>, Bridelia <dbl>,
#   Burkea <dbl>, Byrsocarpus <dbl>,

In [ ]:
# Save data in drive
%%R
write.csv(cluster7_mean, "Data/cluster7_abundance_mean.csv")
write.csv(cluster7_median, "Data/cluster7_abundance_median.csv")
write.csv(cluster7_sd, "Data/cluster7_abundance_sd.csv")